# Distances Between Observations

Read this notebook from top to bottom and fill in the code as you go. Work together and discuss with other students in the class. Try to resolve any errors on your own first, but don't get stuck; ask for help!

In addition to writing and running code, be sure to examine any output and interpret the results before moving on.

For many of these questions, there are several approaches, and there is no single right answer. You should try a few different things and compare with your classmates.

We will use `scikit-learn` extensively later, but for this activity you might want to stick with `pandas`.


In [23]:
import pandas as pd
import numpy as np

## Ames - Recommending Similar Homes

1\. Suppose that you really like house 0 in the Ames housing data set, but it is too expensive. Find cheaper homes that are similar to it --- in terms of living area, number of bedrooms, number of bathrooms --- by calculating distances from house 0. You might want to try different distance metrics and different scaling methods; how sensitive are your results to these choices?

Be sure to actually look at the profiles of the homes that your algorithm picked out as most similar (based on these 3 variables). Do they make sense?

_Think:_ If the goal is to find a "good deal" on a similar house, should sale price be included as a variable in your distance metric?

In [24]:
ames_df = pd.read_csv("https://raw.githubusercontent.com/wblakecannon/ames/master/data/housing.csv")
ames_df.head()

,Unnamed: 0,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
0,0,1,526301100,20,RL,141.0,31770,Pave,NaN,IR1,...,0,NaN,NaN,NaN,0,5,2010,WD,Normal,215000
1,1,2,526350040,20,RH,80.0,11622,Pave,NaN,Reg,...,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal,105000
2,2,3,526351010,20,RL,81.0,14267,Pave,NaN,IR1,...,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal,172000
3,3,4,526353030,20,RL,93.0,11160,Pave,NaN,Reg,...,0,NaN,NaN,NaN,0,4,2010,WD,Normal,244000
4,4,5,527105010,60,RL,74.0,13830,Pave,NaN,IR1,...,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal,189900


In [25]:
features = ['Gr Liv Area', 'Bedroom AbvGr', 'Full Bath', 'Half Bath']
house0 = ames_df[features].iloc[0]
house0

Gr Liv Area      1656
Bedroom AbvGr       3
Full Bath           1
Half Bath           0
Name: 0, dtype: int64

In [26]:
# Euclidean distance with z score scaling
means = ames_df[features].mean()
stds = ames_df[features].std()

df_scaled = (ames_df[features] - means) / stds
house0_scaled = (house0 - means) / stds

ames_df['dist_eucl'] = np.sqrt(((df_scaled - house0_scaled) ** 2).sum(axis=1))

cheaper = ames_df[ames_df['SalePrice'] < ames_df['SalePrice'].iloc[0]].copy()
cheaper.nsmallest(5, 'dist_eucl')[features + ['SalePrice', 'dist_eucl']]

,Gr Liv Area,Bedroom AbvGr,Full Bath,Half Bath,SalePrice,dist_eucl
1226,1661,3,1,0,165500,0.009891
1940,1647,3,1,0,153000,0.017804
291,1666,3,1,0,100000,0.019782
758,1666,3,1,0,135000,0.019782
1357,1666,3,1,0,161000,0.019782


In [27]:
# Manhattan distance with z score scaling
ames_df['dist_man'] = ((df_scaled - house0_scaled).abs()).sum(axis=1)
cheaper = ames_df[ames_df['SalePrice'] < ames_df['SalePrice'].iloc[0]].copy()

cheaper.nsmallest(5, 'dist_man')[features + ['SalePrice', 'dist_man']]

,Gr Liv Area,Bedroom AbvGr,Full Bath,Half Bath,SalePrice,dist_man
1226,1661,3,1,0,165500,0.009891
1940,1647,3,1,0,153000,0.017804
291,1666,3,1,0,100000,0.019782
758,1666,3,1,0,135000,0.019782
1357,1666,3,1,0,161000,0.019782


Both metrics (Euclidean and Manhattan distance on standardized data) returned the same top 5 most similar houses to house 0. Standardization is necessary because Gr Liv Area would dominate other features in the distance calculation. The most similar house is almost identical to house 0 but $50k cheaper, which makes sense. Sale price should not be included in the distance metric because the goal is to find structurally similar homes that are also cheaper, not homes at a similar price point.

2\. Continuing part 1. Suppose that you really like house 0 in the data set, but it is too expensive. Find cheaper homes that are similar to it --- in terms of living area, number of bedrooms, number of bathrooms, **and House Style** --- by calculating distances from house 0. You might want to try different distance metrics and different scaling methods; how sensitive are your results to these choices?

Be sure to actually look at the profiles of the homes that your algorithm picked out as most similar. Do they make sense?

In [28]:
# One-hot encode House Style
style_dummies = pd.get_dummies(ames_df['House Style'], dtype=int)
style_dummies.head()

,1.5Fin,1.5Unf,1Story,2.5Fin,2.5Unf,2Story,SFoyer,SLvl
0,0,0,1,0,0,0,0,0
1,0,0,1,0,0,0,0,0
2,0,0,1,0,0,0,0,0
3,0,0,1,0,0,0,0,0
4,0,0,0,0,0,1,0,0


In [29]:
# Combine with quantitative features
features_quant = ['Gr Liv Area', 'Bedroom AbvGr', 'Full Bath', 'Half Bath']
df_combined = pd.concat([ames_df[features_quant], style_dummies], axis=1)

# Standardize quant features only
for col in features_quant:
    df_combined[col] = (df_combined[col] - df_combined[col].mean()) / df_combined[col].std()

house0 = df_combined.iloc[0]

df_combined.head()

,Gr Liv Area,Bedroom AbvGr,Full Bath,Half Bath,1.5Fin,1.5Unf,1Story,2.5Fin,2.5Unf,2Story,SFoyer,SLvl
0,0.309212,0.176064,-1.024618,-0.755074,0,0,1,0,0,0,0,0
1,-1.194223,-1.032058,-1.024618,-0.755074,0,0,1,0,0,0,0,0
2,-0.337661,0.176064,-1.024618,1.234464,0,0,1,0,0,0,0,0
3,1.207317,0.176064,0.783894,1.234464,0,0,1,0,0,0,0,0
4,0.255801,0.176064,0.783894,1.234464,0,0,0,0,0,1,0,0


In [30]:
# Euclidean distance
ames_df['dist_euc'] = np.sqrt(((df_combined - house0) ** 2).sum(axis=1))

cheaper = ames_df[ames_df['SalePrice'] < ames_df['SalePrice'].iloc[0]].copy()
cheaper.nsmallest(5, 'dist_euc')[features_quant + ['House Style', 'SalePrice', 'dist_euc']]

,Gr Liv Area,Bedroom AbvGr,Full Bath,Half Bath,House Style,SalePrice,dist_euc
1940,1647,3,1,0,1Story,153000,0.017804
618,1644,3,1,0,1Story,167000,0.023738
2700,1640,3,1,0,1Story,131000,0.031651
314,1687,3,1,0,1Story,160000,0.061324
788,1689,3,1,0,1Story,127500,0.065281


In [31]:
# Manhattan distance
ames_df['dist_man'] = (df_combined - house0).abs().sum(axis=1)

cheaper = ames_df[ames_df['SalePrice'] < ames_df['SalePrice'].iloc[0]].copy()
cheaper.nsmallest(5, 'dist_man')[features_quant + ['House Style', 'SalePrice', 'dist_man']]

,Gr Liv Area,Bedroom AbvGr,Full Bath,Half Bath,House Style,SalePrice,dist_man
1940,1647,3,1,0,1Story,153000,0.017804
618,1644,3,1,0,1Story,167000,0.023738
2700,1640,3,1,0,1Story,131000,0.031651
314,1687,3,1,0,1Story,160000,0.061324
788,1689,3,1,0,1Story,127500,0.065281


After adding House Style and one-hot encoding it, both metrics again returned the same top 5. They were all "1Story" homes matching house 0. Same as before, I had to standardize the data frame before calculating distances or else Gr Liv Area would drastically outweigh other features. Looking at the features, the houses it returned make sense as they are very similar to house0, just cheaper. One thing to note is that one-hot encoding creates 8 binary columns for style vs. 4 numeric features, so style may have more weight in the distance calculation.

3\. Continuing parts 1 and 2. Suppose that you really like house 0 in the data set, but it is too expensive. Find cheaper homes that are similar to it, by calculating distances. You can **choose the variables to include, but include both quantitative and categorical variables**. Be sure to actually look at the profiles of the homes that your algorithm picked out as most similar. Do they make sense?

You might want to try different distance metrics and different scaling methods; how sensitive are your results to these choices?

_Hint:_ There are many variables in the data set. Do not attempt to compute distance based on all the variables! You will want to pare down the number of variables, but be sure to include a mixture of categorical and quantitative variables. Refer to the [data documentation](https://ww2.amstat.org/publications/jse/v19n3/decock/DataDocumentation.txt) for information about the variables.


In [37]:
features_quant = ['Gr Liv Area', 'Bedroom AbvGr', 'Full Bath', 'Half Bath', 
                  'Overall Qual', 'Garage Area', 'Year Built']
features_cat = ['House Style', 'Neighborhood', 'Bldg Type']

ames_df[features_quant + features_cat + ['SalePrice']].iloc[0]

Gr Liv Area        1656
Bedroom AbvGr         3
Full Bath             1
Half Bath             0
Overall Qual          6
Garage Area       528.0
Year Built         1960
House Style      1Story
Neighborhood      NAmes
Bldg Type          1Fam
SalePrice        215000
Name: 0, dtype: object

In [38]:
# One-hot encode cat features
cat_dummies = pd.get_dummies(ames_df[features_cat], dtype=int)

# Combine with quant features
df_combined = pd.concat([ames_df[features_quant], cat_dummies], axis=1)

# Z-score scale quantitative features only
for col in features_quant:
    df_combined[col] = (df_combined[col] - df_combined[col].mean()) / df_combined[col].std()

house0 = df_combined.iloc[0]

In [39]:
# Euclidean distance
ames_df['dist_euc'] = np.sqrt(((df_combined - house0) ** 2).sum(axis=1))

cheaper = ames_df[ames_df['SalePrice'] < ames_df['SalePrice'].iloc[0]].copy()
cheaper.nsmallest(5, 'dist_euc')[features_quant + features_cat + ['SalePrice', 'dist_euc']]

,Gr Liv Area,Bedroom AbvGr,Full Bath,Half Bath,Overall Qual,Garage Area,Year Built,House Style,Neighborhood,Bldg Type,SalePrice,dist_euc
1240,1570,3,1,0,6,441.0,1958,1Story,NAmes,1Fam,166800,0.443832
1896,1429,3,1,0,6,572.0,1960,1Story,NAmes,1Fam,181900,0.493469
618,1644,3,1,0,6,418.0,1953,1Story,NAmes,1Fam,167000,0.561941
989,1414,3,1,0,6,601.0,1958,1Story,NAmes,1Fam,176500,0.590580
1216,1252,3,1,0,6,528.0,1959,1Story,NAmes,1Fam,142000,0.799878


In [40]:
# Manhattan distance
ames_df['dist_man'] = (df_combined - house0).abs().sum(axis=1)

cheaper = ames_df[ames_df['SalePrice'] < ames_df['SalePrice'].iloc[0]].copy()
cheaper.nsmallest(5, 'dist_man')[features_quant + features_cat + ['SalePrice', 'dist_man']]

,Gr Liv Area,Bedroom AbvGr,Full Bath,Half Bath,Overall Qual,Garage Area,Year Built,House Style,Neighborhood,Bldg Type,SalePrice,dist_man
1240,1570,3,1,0,6,441.0,1958,1Story,NAmes,1Fam,166800,0.640815
1896,1429,3,1,0,6,572.0,1960,1Story,NAmes,1Fam,181900,0.653659
618,1644,3,1,0,6,418.0,1953,1Story,NAmes,1Fam,167000,0.766696
1216,1252,3,1,0,6,528.0,1959,1Story,NAmes,1Fam,142000,0.832258
989,1414,3,1,0,6,601.0,1958,1Story,NAmes,1Fam,176500,0.884313


I added Overall Qual, Garage Area, Year Built, Neighborhood, and Bldg Type on top of the previous features. Both Euclidean and Manhattan returned the same top 5, with only a slight reordering (houses 1216 and 989 swapped). All 5 matches are 1Story, 1Fam homes in the NAmes neighborhood built in the late 1950s, which lines up with house 0. The results make sense since they match on style, location, building type, and era, and are all cheaper. Adding more categorical variables means even more one-hot columns, so the categorical features could outweigh the numeric ones in the distance. The results weren't very sensitive to the choice of metric here, but scaling still matters for the same reasons as before.

## Colleges similar to Cal Poly

We'll use data from the [College Scorecard data](https://collegescorecard.ed.gov/) to find colleges and universities that are similar to Cal Poly.

In [42]:
df_college = pd.read_csv("https://datasci112.stanford.edu/data/college_attributes.csv")

df_college.set_index("Institution", inplace = True)

df_college.head()

,City,State,AdmissionRate,Undergraduates,CarnegieClassification,Ownership,PCIP01,PCIP03,PCIP04,PCIP05,...,PCIP44,PCIP45,PCIP46,PCIP47,PCIP48,PCIP49,PCIP50,PCIP51,PCIP52,PCIP54
Institution,,,,,,,,,,,,,,,,,,,,,
Alabama A & M University,Normal,AL,0.7160,5098.0,Master's Colleges & Universities: Larger Programs,Public,0.0445,0.0071,0.0053,0.0000,...,0.0409,0.0249,0.0,0.0,0.0,0.0,0.0231,0.0000,0.1637,0.0000
University of Alabama at Birmingham,Birmingham,AL,0.8854,13284.0,Doctoral Universities: Very High Research Acti...,Public,0.0000,0.0000,0.0000,0.0020,...,0.0195,0.0239,0.0,0.0,0.0,0.0,0.0249,0.2088,0.2159,0.0141
University of Alabama in Huntsville,Huntsville,AL,0.7367,7358.0,Doctoral Universities: Very High Research Acti...,Public,0.0000,0.0000,0.0000,0.0000,...,0.0000,0.0127,0.0,0.0,0.0,0.0,0.0407,0.1341,0.1930,0.0073
Alabama State University,Montgomery,AL,0.9799,3495.0,Doctoral/Professional Universities,Public,0.0000,0.0000,0.0000,0.0000,...,0.0648,0.0196,0.0,0.0,0.0,0.0,0.0511,0.0904,0.1513,0.0059
The University of Alabama,Tuscaloosa,AL,0.7890,30725.0,Doctoral Universities: Very High Research Acti...,Public,0.0000,0.0061,0.0000,0.0019,...,0.0072,0.0661,0.0,0.0,0.0,0.0,0.0234,0.1077,0.2916,0.0096


We'll want to single out Cal Poly, which we can do like this.

In [43]:
school_name = "California Polytechnic State University-San Luis Obispo"

cp = df_college.loc[school_name]

cp

City                                                        San Luis Obispo
State                                                                    CA
AdmissionRate                                                          0.33
Undergraduates                                                      21090.0
CarnegieClassification    Master's Colleges & Universities: Larger Programs
Ownership                                                            Public
PCIP01                                                               0.1084
PCIP03                                                               0.0255
PCIP04                                                               0.0441
PCIP05                                                               0.0019
PCIP09                                                               0.0353
PCIP10                                                               0.0175
PCIP11                                                               0.0326
PCIP12      

1\. Based on only the admission rate and the number of undergraduates, what schools are most similar to Cal Poly? Specify how you're making this decision.

In [ ]:
# Features for comparison
features = ['AdmissionRate', 'Undergraduates']

# Drop rows with missing values in these columns
df_clean = df_college[features].dropna()

cp_vals = df_clean.loc[school_name]
cp_vals

AdmissionRate         0.33
Undergraduates    21090.00
Name: California Polytechnic State University-San Luis Obispo, dtype: float64

In [48]:
# Z-score scaling
means = df_clean.mean()
stds = df_clean.std()

df_scaled = (df_clean - means) / stds
cp_scaled = (cp_vals - means) / stds

# Euclidean distance
distances = np.sqrt(((df_scaled - cp_scaled) ** 2).sum(axis=1))
df_college['dist'] = distances

df_college.nsmallest(5, 'dist')[features + ['dist']]

,AdmissionRate,Undergraduates,dist
Institution,,,
California Polytechnic State University-San Luis Obispo,0.3300,21090.0,0.000000
University of California-Santa Barbara,0.2918,23081.0,0.309162
DeVry University-Illinois,0.4552,19729.0,0.593121
University of North Carolina at Chapel Hill,0.2040,19722.0,0.596846
Clemson University,0.4922,21577.0,0.736788


Using Euclidean distance with z-score scaling on admission rate and number of undergraduates, the most similar schools to Cal Poly are UCSB, DeVry, UNC Chapel Hill, and Clemson. Scaling is needed because undergraduates is in the tens of thousands while admission rate is between 0 and 1.

2\. Now consider the admission rate, the number of undergraduates, and also the [Carnegie classification](https://en.wikipedia.org/wiki/Carnegie_Classification_of_Institutions_of_Higher_Education) of the type of school, and the ownership (public, private, etc.) Based on these variables, what schools are most similar to Cal Poly? Specify how you're making this decision.

In [47]:
features_quant = ['AdmissionRate', 'Undergraduates']
features_cat = ['CarnegieClassification', 'Ownership']

df_college.loc[school_name, features_quant + features_cat]

AdmissionRate                                                          0.33
Undergraduates                                                      21090.0
CarnegieClassification    Master's Colleges & Universities: Larger Programs
Ownership                                                            Public
Name: California Polytechnic State University-San Luis Obispo, dtype: object

In [49]:
df_clean = df_college[features_quant + features_cat].dropna()

# One-hot encode cat features
cat_dummies = pd.get_dummies(df_clean[features_cat], dtype=int)
df_combined = pd.concat([df_clean[features_quant], cat_dummies], axis=1)

# Z-score scale for quant features
for col in features_quant:
    df_combined[col] = (df_combined[col] - df_combined[col].mean()) / df_combined[col].std()

cp_combined = df_combined.loc[school_name]

# Euclidean distance
df_college['dist2'] = np.sqrt(((df_combined - cp_combined) ** 2).sum(axis=1))

df_college.nsmallest(5, 'dist2')[features_quant + features_cat + ['dist2']]

,AdmissionRate,Undergraduates,CarnegieClassification,Ownership,dist2
Institution,,,,,
California Polytechnic State University-San Luis Obispo,0.3300,21090.0,Master's Colleges & Universities: Larger Programs,Public,0.000000
CUNY Hunter College,0.4590,17293.0,Master's Colleges & Universities: Larger Programs,Public,0.761441
CUNY Bernard M Baruch College,0.5056,15483.0,Master's Colleges & Universities: Larger Programs,Public,1.073601
CUNY John Jay College of Criminal Justice,0.4458,12834.0,Master's Colleges & Universities: Larger Programs,Public,1.184989
CUNY Brooklyn College,0.5136,12567.0,Master's Colleges & Universities: Larger Programs,Public,1.376321


After adding Carnegie classification and Ownership via one-hot encoding, the top matches are all CUNY schools. They all match Cal Poly's classification (Master's Colleges, Larger Programs) and are public, which makes the results more meaningful because the categorical variables help filter to schools that are actually similar in type.

3\. The columns whose names begin with "PCIP" contain the proportions of students at each school studying various fields (e.g., Engineering, Psychology). Each field is represented by a two-digit code called the [CIP code](https://nces.ed.gov/ipeds/cipcode/browse.aspx?y=55).

If we only consider the proportions of students studying various fields, what schools are most similar to Cal Poly? Specify how you're making this decision.

In [51]:
# Get all PCIP columns (proportions of students in each field)
pcip_cols = [col for col in df_college.columns if col.startswith('PCIP')]

cp_pcip = df_college.loc[school_name, pcip_cols]
cp_pcip.sort_values(ascending=False).head(10)

PCIP14    0.2314
PCIP52    0.1637
PCIP01    0.1084
PCIP45    0.0588
PCIP26    0.0495
PCIP04    0.0441
PCIP31    0.0419
PCIP09    0.0353
PCIP11    0.0326
PCIP42    0.0288
Name: California Polytechnic State University-San Luis Obispo, dtype: object

In [57]:
df_pcip = df_college[pcip_cols].dropna()

# Scale
means = df_pcip.mean()
stds = df_pcip.std()
df_pcip_scaled = (df_pcip - means) / stds
cp_scaled = (cp_vals - means) / stds

df_college['dist_pcip_std'] = np.sqrt(((df_pcip_scaled - cp_scaled) ** 2).sum(axis=1))

df_college.nsmallest(5, 'dist_pcip_std')[pcip_cols[:5] + ['dist_pcip_std']]

,PCIP01,PCIP03,PCIP04,PCIP05,PCIP09,dist_pcip_std
Institution,,,,,,
California Polytechnic State University-San Luis Obispo,0.1084,0.0255,0.0441,0.0019,0.0353,0.000000
Iowa State University,0.1013,0.0131,0.0163,0.0009,0.0392,1.986207
California State Polytechnic University-Pomona,0.0384,0.0000,0.0324,0.0043,0.0306,2.241726
Texas A & M University-College Station,0.0847,0.0241,0.0111,0.0001,0.0397,2.258942
Mississippi State University,0.0619,0.0263,0.0208,0.0000,0.0360,2.335288


Using Euclidean distance with z-score scaling on the PCIP columns, the most similar schools to Cal Poly are Iowa State, Cal Poly Pomona, Texas A&M, and Mississippi State. I z-score scaled each PCIP column, then computed Euclidean distance between Cal Poly's row and every other school's row, and picked the smallest distances.